## READ ME FIRST

This notebook produces a weekly release from the `develop` branch of Cytoscape.

The weekly release consists of Install4j generated installers that can be put on a public server to be downloaded by testers. This script will NOT push any code changes or resources to GitHub or Nexus or update any additional resources like the manual or API docs. 

This notebook is configured for the build server on which it is run. Notes have been made where local environment variables have been set.

Before starting this process, it is a good idea to Restart and Clear Output for this notebook's Kernel.


### 1. Set up Build Environment

This sets up the build environment, including Java and Maven versions, and the root directory of the build.


In [1]:
from subprocess import Popen, PIPE
import os
import shutil

Set the notebook directory. Note that this is system dependent.

In [2]:
NOTEBOOK_DIR = '/home/cybuilder/cytoscape-admin-scripts'

print(NOTEBOOK_DIR)

/home/cybuilder/cytoscape-admin-scripts


Set the Java environment. This is Java 11 for Cytoscape 3.8.0 and above.

Note that this is system dependent, and may need to be changed to reflect the system's configuration.

In [3]:
%env JAVA_HOME=/opt/jdk-17

env: JAVA_HOME=/opt/jdk-17


Set the MAVEN_HOME environment variable, as well as point the path to the correct Maven binaries. 

Note that this is system dependent and may need to be changed to reflect the system's configuration.

In [4]:
%env MAVEN_HOME=/opt/maven
%env PATH=/opt/jdk-17/bin:/opt/apache-maven-3.6.0/bin/:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin

env: MAVEN_HOME=/opt/maven
env: PATH=/opt/jdk-17/bin:/opt/apache-maven-3.6.0/bin/:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin


Prepare the target directory and clone the cytoscape git repo.

In [5]:
print('Changing to directory: ' + NOTEBOOK_DIR)
os.chdir(NOTEBOOK_DIR)

# Point to build location (the directory to clone parent cytoscape into)
BUILD_PARENT_DIR = os.path.join(os.getcwd(), 'release-build')
if not os.path.exists(BUILD_PARENT_DIR):
    os.mkdir(BUILD_PARENT_DIR)
else:
    shutil.rmtree(BUILD_PARENT_DIR)
    os.mkdir(BUILD_PARENT_DIR)

os.chdir(BUILD_PARENT_DIR)
![[ -d cytoscape ]] || git clone https://github.com/cytoscape/cytoscape
CYTOSCAPE_ROOT_DIR = os.path.join(BUILD_PARENT_DIR, 'cytoscape')
CYTOSCAPE_DIR = os.path.join(CYTOSCAPE_ROOT_DIR, 'cytoscape')

def cd(directory=BUILD_PARENT_DIR, *subdirs):
    if subdirs:
        directory = os.path.join(directory, *subdirs)
    if os.getcwd() != directory:
        os.chdir(directory)

Changing to directory: /home/cybuilder/cytoscape-admin-scripts
Cloning into 'cytoscape'...
remote: Enumerating objects: 658, done.
remote: Counting objects: 100% (81/81), done.
remote: Compressing objects: 100% (74/74), done.
remote: Total 658 (delta 51), reused 23 (delta 7), pack-reused 577
Receiving objects: 100% (658/658), 788.22 KiB | 0 bytes/s, done.
Resolving deltas: 100% (335/335), done.


In [6]:
STARTING_BRANCH = 'develop'     # develop for major release, release/3.X.X for minor release

## 2. Pull the develop branch of Cytoscape

Note that to execute the checkout step here, you will need to have set up an SSH key on this machine that does not use password validation, or else parts of the build will be stall when they require input. 

In [7]:
cd(CYTOSCAPE_ROOT_DIR)
![[ -d cytoscape ]] || ./cy.sh init
cd(CYTOSCAPE_DIR)
!./cy.sh run-all "git checkout {STARTING_BRANCH}"

Target directory = 
Cytoscape project will be cloned to: /home/cybuilder/cytoscape-admin-scripts/release-build/cytoscape
Cloning into 'cytoscape'...
remote: Enumerating objects: 658, done.
remote: Counting objects: 100% (81/81), done.
remote: Compressing objects: 100% (74/74), done.
remote: Total 658 (delta 51), reused 23 (delta 7), pack-reused 577
Receiving objects: 100% (658/658), 788.22 KiB | 0 bytes/s, done.
Resolving deltas: 100% (335/335), done.
Cloning: parent (URI = git@github.com:cytoscape/cytoscape-parent.git)
Cloning into 'parent'...
remote: Enumerating objects: 1718, done.
remote: Counting objects: 100% (78/78), done.
remote: Compressing objects: 100% (62/62), done.
remote: Total 1718 (delta 52), reused 42 (delta 16), pack-reused 1640
Receiving objects: 100% (1718/1718), 265.85 KiB | 0 bytes/s, done.
Resolving deltas: 100% (680/680), done.
~/cytoscape-admin-scripts/release-build/cytoscape/cytoscape/parent ~/cytoscape-admin-scripts/release-build/cytoscape/cytoscape
Already o

## 2a. Reset

In [8]:
cd(CYTOSCAPE_DIR)
!./cy.sh run-all 'git clean -f -d'
!./cy.sh run-all 'git reset --hard'

------------------------------------------------------------------------
Executing command: git clean -f -d
--in .
------------------------------------------------------------------------
--in parent
------------------------------------------------------------------------
--in api
------------------------------------------------------------------------
--in impl
------------------------------------------------------------------------
--in support
------------------------------------------------------------------------
--in gui-distribution
------------------------------------------------------------------------
--in app-developer
------------------------------------------------------------------------
------------------------------------------------------------------------
Executing command: git reset --hard
--in .
HEAD is now at 9814970 From now on, development branch is for 3.11.0-SNAPSHOT.
------------------------------------------------------------------------
--in parent
HEAD is n

## 2. Verify status

In [9]:
cd(CYTOSCAPE_DIR)
!./cy.sh run-all 'git status'

------------------------------------------------------------------------
Executing command: git status
--in .
# On branch develop
nothing to commit, working directory clean
------------------------------------------------------------------------
--in parent
# On branch develop
nothing to commit, working directory clean
------------------------------------------------------------------------
--in api
# On branch develop
nothing to commit, working directory clean
------------------------------------------------------------------------
--in impl
# On branch develop
nothing to commit, working directory clean
------------------------------------------------------------------------
--in support
# On branch develop
nothing to commit, working directory clean
------------------------------------------------------------------------
--in gui-distribution
# On branch develop
nothing to commit, working directory clean
------------------------------------------------------------------------
--in app

## 3. Build Cytoscape and ensure no errors

This may take a while. Expect to build subrepos first before building from the root directory

Currently, you should expect two or so non-fatal javadoc-bundle-options related errors.

In [10]:
cd(CYTOSCAPE_DIR)
with open('build_output.txt', 'w') as outf:
    process = Popen('mvn clean install -Dmaven.test.skip=true'.split(' '), 
                stdout=outf,
                cwd=CYTOSCAPE_DIR)
    process.wait()

print("Showing ERROR lines in build...")
!cat build_output.txt | grep ERROR

Showing ERROR lines in build...
[ERROR] Error fetching link: /home/cybuilder/cytoscape-admin-scripts/release-build/cytoscape/cytoscape/api/app-api/target/javadoc-bundle-options. Ignored it.
[ERROR] Error fetching link: /home/cybuilder/cytoscape-admin-scripts/release-build/cytoscape/cytoscape/api/swing-app-api/target/javadoc-bundle-options. Ignored it.


## 4. Build Cytoscape installers

This requires Install4J to be configured on your machine and for the Install4j project file to point to the correct Mac signing file. The keystore password is set below (blank in this config), to be interpreted by the Install4j Maven plugin.

In [11]:
cd(CYTOSCAPE_DIR, 'gui-distribution', 'packaging')
%env MAC_KEYSTORE_PASSWORD=
!mvn clean install -U

env: MAC_KEYSTORE_PASSWORD=
[INFO] Scanning for projects...
[WARNING] 
[WARNING] Some problems were encountered while building the effective model for org.cytoscape.distribution:packaging:jar:3.11.0-SNAPSHOT
[WARNING] 'build.plugins.plugin.version' for org.sonatype.install4j:install4j-maven-plugin is missing. @ org.cytoscape.distribution:packaging:[unknown-version], /home/cybuilder/cytoscape-admin-scripts/release-build/cytoscape/cytoscape/gui-distribution/packaging/pom.xml, line 138, column 15
[WARNING] 
[WARNING] It is highly recommended to fix these problems because they threaten the stability of your build.
[WARNING] 
[WARNING] For this reason, future Maven versions might no longer support building such malformed projects.
[WARNING] 
[INFO] 
[INFO] ----------------< org.cytoscape.distribution:packaging >----------------
[INFO] Building Cytoscape Release Packaging 3.11.0-SNAPSHOT
[INFO] --------------------------------[ jar ]---------------------------------
Downloaded from central: 

[INFO]     Compiling launcher 'Cytoscape':
[INFO]       Signing launcher
[INFO]   Creating media file: 
[INFO]     Signing launcher
[INFO]     Zipping custom code & resources JAR file
[INFO]     Identifying components
[INFO]     Shrinking runtime
[INFO]   
[INFO] Creating media file 'macos aarch64':
[INFO]   Collecting files:
[INFO]   Compiling launchers:
[INFO]     Compiling launcher 'Cytoscape':
[INFO]       Signing launcher
[INFO]   Creating media file: 
[INFO]     Signing launcher
[INFO]     Zipping custom code & resources JAR file
[INFO]     Identifying components
[INFO]     Shrinking runtime
[INFO]   
[INFO] Creating media file 'unix':
[INFO]   Collecting files:
[INFO]   Compiling launchers:
[INFO]     Compiling launcher 'Cytoscape':
[INFO]       Generating launcher script file
[INFO]   Creating media file: 
[INFO]     Generating launcher script file
[INFO]     Zipping custom code & resources JAR file
[INFO]     Identifying components
[INFO]     Shrinking runtime
[INFO]   
[INFO]

[INFO] Installing /home/cybuilder/cytoscape-admin-scripts/release-build/cytoscape/cytoscape/gui-distribution/packaging/target/packaging-3.11.0-SNAPSHOT-sources.jar to /home/cybuilder/.m2/repository/org/cytoscape/distribution/packaging/3.11.0-SNAPSHOT/packaging-3.11.0-SNAPSHOT-sources.jar
[INFO] ------------------------------------------------------------------------
[INFO] BUILD SUCCESS
[INFO] ------------------------------------------------------------------------
[INFO] Total time:  55.374 s
[INFO] Finished at: 2023-04-17T09:22:41-07:00
[INFO] ------------------------------------------------------------------------


## 5. Copying Cytoscape installers to weekly download page

When completed installer executables can be found in `cytoscape/cytoscape/gui-distribution/packaging/target/media` and compressed builds for Linux and Windows can be found in `cytoscape/cytoscape/gui-distribution/assembly/target`

all of which should be copied to `/var/www/html/cytoscape-builds/Cytoscape-3.9.0/<DATE>`



In [12]:
!mkdir -p /var/www/html/cytoscape-builds/Cytoscape-3.11.0/CYTOSCAPE-13071
!rsync -av --progress /home/cybuilder/cytoscape-admin-scripts/release-build/cytoscape/cytoscape/gui-distribution/packaging/target/media/* /var/www/html/cytoscape-builds/Cytoscape-3.11.0/CYTOSCAPE-13071
!rsync -av --progress /home/cybuilder/cytoscape-admin-scripts/release-build/cytoscape/cytoscape/gui-distribution/assembly/target/*.{gz,zip} /var/www/html/cytoscape-builds/Cytoscape-3.11.0/CYTOSCAPE-13071


sending incremental file list
Cytoscape_3_11_0-SNAPSHOT_macos_aarch64.dmg
    229,369,115 100%  242.74MB/s    0:00:00 (xfr#1, to-chk=7/8)
Cytoscape_3_11_0-SNAPSHOT_macos_x64.dmg
    231,974,167 100%  131.29MB/s    0:00:01 (xfr#2, to-chk=6/8)
Cytoscape_3_11_0-SNAPSHOT_unix.sh
    235,233,270 100%  134.82MB/s    0:00:01 (xfr#3, to-chk=5/8)
Cytoscape_3_11_0-SNAPSHOT_windows_64bit.exe
    229,913,088 100%  159.58MB/s    0:00:01 (xfr#4, to-chk=4/8)
md5sums
            298 100%    0.78kB/s    0:00:00 (xfr#5, to-chk=3/8)
output.txt
            812 100%    2.11kB/s    0:00:00 (xfr#6, to-chk=2/8)
sha256sums
            426 100%    1.11kB/s    0:00:00 (xfr#7, to-chk=1/8)
updates.xml
          1,750 100%    4.56kB/s    0:00:00 (xfr#8, to-chk=0/8)

sent 926,719,712 bytes  received 168 bytes  205,937,751.11 bytes/sec
total size is 926,492,926  speedup is 1.00
sending incremental file list
cytoscape-unix-3.11.0-SNAPSHOT.tar.gz
    343,369,480 100%  214.15MB/s    0:00:01 (xfr#1, to-chk=1/2)
cytoscape

## 6. Notarizing on idekerlab-macmini

The install4j generated .dmg must be notarized to run on macOS 10.15 and above. The .dmg will be sent to idekerlab-macmini and submitted for notarization. An email will be sent to William Markuske when the notarization process is complete. As soon as the .dmg is notarized it will work and no further action is needed.

In [13]:
#!rsync -av --progress /var/www/html/cytoscape-builds/Cytoscape-3.9.0/$(date +%Y_%m_%d)/*.dmg idekerlab@idekerlab-macmini.ucsd.edu:~/apps_to_notarize/
#!ssh idekerlab@idekerlab-macmini.ucsd.edu '~/notarizedmg_jing.sh ~/apps_to_notarize/*.dmg Snapshot.$(date +%Y%m%d)'    

building file list ... 
1 file to consider
Cytoscape_3_9_0-SNAPSHOT_macos.dmg
    292,894,009 100%   84.61MB/s    0:00:03 (xfr#1, to-chk=0/1)

sent 292,929,916 bytes  received 46 bytes  65,095,547.11 bytes/sec
total size is 292,894,009  speedup is 1.00
No errors uploading '/Users/idekerlab/apps_to_notarize/Cytoscape_3_9_0-SNAPSHOT_macos.dmg'.
RequestUUID = 6a41ff2a-9fd8-43d6-b378-adfd075b613b




You can check the status of the notarization by pasting in the `RequestUUID` value into the following command and running.

In [16]:
#!ssh idekerlab@idekerlab-macmini.ucsd.edu '~/notarizestatus_jing.sh 6a41ff2a-9fd8-43d6-b378-adfd075b613b'

No errors getting notarization info.

          Date: 2021-04-30 13:56:52 +0000
          Hash: 7d32053845cd00b39f1a454710f3a650974f7f5d980ea41753a515efb27ee8cc
    LogFileURL: https://osxapps-ssl.itunes.apple.com/itunes-assets/Enigma115/v4/91/1f/c3/911fc3d0-a26b-067b-633c-b9cc8ce3e71a/developer_log.json?accessKey=1619985924_8906453629669302967_JpRk3DbwQdUPeY0fjgXAfuB0rVc%2FQkGm0wHltInIzu7k9XgkuwZmK3DtsDmyD%2FHD%2FAP1BLjqG%2BlEIEvOJNlbOcYPq%2FKsH%2B5LfUd%2Byja0avqDGmZ15ZgRo4Rp9sOJYsf%2Fv8PydAzkQSHl9NQDC8CfSUg%2BE1%2Fpw47CEbwJpnrmmfQ%3D
   RequestUUID: 6a41ff2a-9fd8-43d6-b378-adfd075b613b
        Status: success
   Status Code: 0
Status Message: Package Approved

